# 04. Template-corrected shares by year and selection route, trend, Table II and Fig. 2

**Design decisions.**
- Inputs: the final labels from notebook 02 for the advertisements in the three pools (2019 subset, rebuilt 2024 and 2026 pools), the full stratum sizes from `reports/pool_sizes.csv`, and the advertisement text from `data/subsets/` for clustering.
- Template clusters with MinHash and LSH on title plus full description (threshold 0.85, 5-word shingles, 128 permutations); cluster bootstrap with B = 20,000 and seed 7 per (year, route) stratum; design effect and effective n; logistic trend of the label on calendar year with cluster-robust standard errors (`src/dedup_inference.py`, the June module unchanged).
- The primary estimate per year is the "title cue" stratum, advertisements whose title carries a junior word, the only definition that exists in every source. The "platform level" stratum (selected only by the platform's experience-level field) is reported beside it. A year-level figure across both routes is a pool-size-weighted combination of the stratum bootstraps, which is what the per-stratum sampling cap requires.
- Outputs: `reports/table2.csv`, `reports/results.json`, `figures/fig2_prevalence.png` (600 dpi, paper names in the axis labels).


In [ ]:
# === SETTINGS ===
import os
from pathlib import Path
REPO_URL   = "https://github.com/anasbiswas1/hiring-research.git"
DRIVE_ROOT = Path(os.environ.get("HIRING_DRIVE_ROOT", "/content/drive/MyDrive/Hiring_Research"))
REPO_DIR   = DRIVE_ROOT / "hiring-research"
MODEL     = "claude-sonnet-5-5"
N_BOOT    = 20000
SEED      = 7
THRESHOLD = 0.85


In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import os, sys, shutil, subprocess, importlib
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
CRED = DRIVE_ROOT / ".secrets"
for name, dot in [("git-credentials", ".git-credentials"), ("gitconfig", ".gitconfig")]:
    if (CRED / name).exists():
        shutil.copy(CRED / name, Path.home() / dot)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)
subprocess.run(["git", "config", "--global", "credential.helper", "store"])
os.environ["HIRING_DRIVE_ROOT"] = str(DRIVE_ROOT)
os.environ["HIRING_REPO_ROOT"] = str(REPO_DIR)
assert (REPO_DIR / ".git").exists(), f"no git checkout at {REPO_DIR}: run notebook 00 Stage 0 first"
os.chdir(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
importlib.invalidate_caches()
pull = subprocess.run(["git", "pull", "origin", "main"], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())
print("repo:", REPO_DIR)


## Stage 0. Install

In [ ]:
# === INSTALL ===
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *['datasketch', 'statsmodels', 'matplotlib', 'pandas']], capture_output=True, text=True)
print("pip -> rc", r.returncode)
print(r.stderr[-600:])


## Stage 1. Template clusters, cluster bootstrap per stratum, weighted year figures, clustered trend

In [ ]:
# === STAGE 1: load, cluster, bootstrap per stratum, weighted year figures, trend ===
import importlib, json
import pandas as pd
import ads, results, dedup_inference
importlib.reload(results); importlib.reload(dedup_inference)
FIN = pd.read_csv(REPO_DIR / "reports" / f"ratings_{MODEL}.csv", dtype={"ad_id": str})
POOL = FIN[FIN.in_pool].copy()
missing = int(POOL.final.isna().sum())
assert missing == 0, f"{missing} advertisements have no final label: finish notebook 02 Stage 4 first"
SUB = REPO_DIR / "data" / "subsets"
ADS = pd.concat([pd.read_csv(p, dtype={"ad_id": str}) for p in sorted(SUB.glob("*.csv"))]).drop_duplicates("ad_id")
SIZES = pd.read_csv(REPO_DIR / "reports" / "pool_sizes.csv")
RES = results.compute(POOL[["ad_id", "year", "why", "final"]], ADS, n_boot=N_BOOT, seed=SEED, threshold=THRESHOLD, pool_sizes=SIZES)
T2 = results.table2(RES)
print(T2.to_string(index=False))
print("\nduplication:"); print(RES["duplication"].to_string(index=False))
print("\ntrend, junior-titled advertisements:", {k: v for k, v in RES["trend_title_cue"].items() if k != "note"})
print("trend, all rated pool advertisements (unweighted):", {k: v for k, v in RES["trend_all_routes_unweighted"].items() if k != "note"})


## Stage 2. Table II, Fig. 2 and results.json

In [ ]:
# === STAGE 2: Table II, Fig. 2 (600 dpi), results.json ===
T2.to_csv(REPO_DIR / "reports" / "table2.csv", index=False)
results.fig2(RES, REPO_DIR / "figures" / "fig2_prevalence.png", dpi=600)
out = results.summary(RES)
out["settings"] = {"model": MODEL, "n_boot": N_BOOT, "seed": SEED, "threshold": THRESHOLD}
ads.write_json(REPO_DIR / "reports" / "results.json", out)
print("written: reports/table2.csv, reports/results.json, figures/fig2_prevalence.png")
from IPython.display import Image, display
display(Image(filename=str(REPO_DIR / "figures" / "fig2_prevalence.png"), width=520))


## End of unit: commit

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
branch = subprocess.run(["git", "symbolic-ref", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
assert branch == "main", f"the repository is on branch {branch!r}, not main: switch to main before committing"
r = subprocess.run(["python", "tools/commit_cell.py", 'nb04: template-corrected shares by year and selection route, pool-size-weighted year figures, clustered trend; reports/table2.csv, results.json; figures/fig2_prevalence.png (600 dpi)'], capture_output=True, text=True)
print(r.stdout)
print(r.stderr)
